# Test Metrics

Evaluate the configured 3-channel and 5-channel `val_IoU` checkpoints on the held-out test split.

## Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import torch
from IPython.display import Markdown, display

CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    raise RuntimeError("Could not find repo root containing src/.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_dataloaders
from src.evaluate import evaluate_test_metrics, load_checkpoint_model
from src.train import sync_class_config
from src.utils import get_device, load_config
from src.visualize import plot_confusion_matrix

device = get_device()
device

## Load Config

In [ ]:
metrics_config = load_config(REPO_ROOT / "configs" / "test_metrics.yaml")
class_names = metrics_config["class_names"]
ignore_index = metrics_config.get("metrics", {}).get("ignore_index", 255)
viz_config = metrics_config.get("visualization", {})
normalize_confusion_matrix = viz_config.get("normalize_confusion_matrix", True)
viz_output_dir = REPO_ROOT / viz_config.get("output_dir", "notebooks/outputs/test_metrics/")
viz_output_dir.mkdir(parents=True, exist_ok=True)

[(entry["name"], entry["label"], entry["checkpoint"]) for entry in metrics_config["models"]]

## Helpers

In [ ]:
def markdown_table(rows, columns):
    header = "| " + " | ".join(label for _, label in columns) + " |"
    divider = "| " + " | ".join("---" for _ in columns) + " |"
    lines = [header, divider]
    for row in rows:
        values = []
        for key, _ in columns:
            value = row.get(key, "")
            if isinstance(value, float):
                value = "nan" if np.isnan(value) else f"{value:.4f}"
            values.append(str(value))
        lines.append("| " + " | ".join(values) + " |")
    display(Markdown("\n".join(lines)))


def evaluate_model_entry(entry):
    train_config = sync_class_config(load_config(REPO_ROOT / entry["config"]))
    train_config["model"]["in_channels"] = entry["in_channels"]
    train_config["model"]["num_classes"] = entry["num_classes"]
    train_config["data"]["channels"] = list(range(1, entry["in_channels"] + 1))
    train_config["data"]["num_classes"] = entry["num_classes"]

    _, _, test_loader = build_dataloaders(train_config["data"])
    model, checkpoint = load_checkpoint_model(
        REPO_ROOT / entry["checkpoint"],
        device=device,
        model_config=train_config["model"],
    )

    metrics = evaluate_test_metrics(
        model=model,
        dataloader=test_loader,
        num_classes=entry["num_classes"],
        device=device,
        ignore_index=ignore_index,
        class_names=class_names,
    )

    return {
        "entry": entry,
        "checkpoint": checkpoint,
        "metrics": metrics,
    }

## Run Evaluation

In [ ]:
results = []
for entry in metrics_config["models"]:
    print(f"Evaluating {entry['label']} from {entry['checkpoint']}")
    results.append(evaluate_model_entry(entry))

len(results)

## Summary Metrics

In [ ]:
summary_rows = []
for result in results:
    entry = result["entry"]
    metrics = result["metrics"]
    summary_rows.append({
        "model": entry["label"],
        "pixels": metrics["num_pixels"],
        "pixel_accuracy": metrics["pixel_accuracy"],
        "mIoU": metrics["mIoU"],
        "mean_dice": metrics["mean_dice"],
        "mAP": metrics["mAP"],
    })

markdown_table(
    summary_rows,
    [
        ("model", "Model"),
        ("pixels", "Pixels"),
        ("pixel_accuracy", "Pixel Acc"),
        ("mIoU", "mIoU"),
        ("mean_dice", "Mean Dice"),
        ("mAP", "mAP"),
    ],
)

## Per-Class Metrics

In [ ]:
for result in results:
    entry = result["entry"]
    display(Markdown(f"### {entry['label']}"))
    markdown_table(
        result["metrics"]["class_rows"],
        [
            ("class_id", "ID"),
            ("class_name", "Class"),
            ("support_pixels", "Support"),
            ("predicted_pixels", "Predicted"),
            ("iou", "IoU"),
            ("dice", "Dice"),
            ("average_precision", "AP"),
        ],
    )

## Confusion Matrices

In [ ]:
for result in results:
    entry = result["entry"]
    fig = plot_confusion_matrix(
        result["metrics"]["confusion_matrix"],
        class_names=class_names,
        normalize=normalize_confusion_matrix,
        title=f"{entry['label']} Confusion Matrix",
    )
    fig.savefig(viz_output_dir / f"{entry['name']}_confusion_matrix.png", dpi=viz_config.get("dpi", 150), bbox_inches="tight")
    display(fig)